# Notebook 01: Download Source Data

Download all raw input files needed for the CAGE x P-body validation pipeline.

**Data Sources:**
1. Hubstenberger et al. 2017 — `mmc3.xlsx` (P-body RNA-seq supplementary)
2. GENCODE v19 — `gencode.v19.annotation.gtf.gz` (gene annotations, hg19)
3. FANTOM5 CAGE — HEK293 expression matrix + peak annotations + sample table
4. DepMap — Common Essential + Strongly Selective gene lists + dependency scores
5. IMPC + MGI — Embryonic lethality + human-mouse ortholog mapping

In [1]:
import os
import requests
import gzip
import shutil
from pathlib import Path
from datetime import datetime

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

def download_file(url, dest, description=""):
    """Download a file if it doesn't already exist."""
    dest = Path(dest)
    if dest.exists():
        print(f"  Already exists: {dest.name} ({dest.stat().st_size:,} bytes)")
        return
    print(f"  Downloading {description or dest.name}...")
    response = requests.get(url, stream=True, timeout=300)
    response.raise_for_status()
    with open(dest, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"  Saved: {dest.name} ({dest.stat().st_size:,} bytes)")

## 1. Hubstenberger et al. 2017 — mmc3.xlsx

Supplementary Table S2 from Molecular Cell paper (DOI: 10.1016/j.molcel.2017.09.003).
Contains raw mapped read counts for 3 P-body and 3 cytosol replicates, hg19-aligned.

In [2]:
print("=== Hubstenberger et al. 2017 ===")
download_file(
    url="https://ars.els-cdn.com/content/image/1-s2.0-S1097276517306512-mmc3.xlsx",
    dest=RAW_DIR / "mmc3.xlsx",
    description="Hubstenberger mmc3.xlsx (P-body RNA-seq data)"
)

=== Hubstenberger et al. 2017 ===
  Already exists: mmc3.xlsx (5,730,333 bytes)


## 2. GENCODE v19 GTF

Comprehensive gene annotation for hg19. Used for:
- Gene biotype classification (protein-coding filter)
- Exonic union gene length calculation
- CAGE peak-to-gene annotation

In [3]:
print("=== GENCODE v19 ===")
download_file(
    url="https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_19/gencode.v19.annotation.gtf.gz",
    dest=RAW_DIR / "gencode.v19.annotation.gtf.gz",
    description="GENCODE v19 GTF annotation (~30 MB compressed)"
)

=== GENCODE v19 ===
  Already exists: gencode.v19.annotation.gtf.gz (37,991,892 bytes)


## 3. FANTOM5 CAGE-seq

Three files needed:
- **Expression matrix** — TPM values per CAGE peak per sample (large file, ~4GB compressed)
- **Peak annotation** — maps peaks to genomic features
- **Sample table** — maps library IDs to cell types (to find HEK293 columns)

The expression matrix is already in TPM. We will identify HEK293 untreated columns
by library ID (e.g., CNhs12328) from the sample table.

In [4]:
print("=== FANTOM5 Sample Table ===")
download_file(
    url="https://fantom.gsc.riken.jp/5/datafiles/latest/extra/Enhancers/Human.sample_name2library_id.txt",
    dest=RAW_DIR / "fantom5_sample_table.txt",
    description="FANTOM5 sample-to-library mapping"
)

=== FANTOM5 Sample Table ===
  Already exists: fantom5_sample_table.txt (115,575 bytes)


In [5]:
# Identify HEK293 samples in FANTOM5
import pandas as pd

sample_table_path = RAW_DIR / "fantom5_sample_table.txt"
if sample_table_path.exists():
    with open(sample_table_path) as f:
        lines = f.readlines()
    
    hek_lines = [l.strip() for l in lines if 'HEK293' in l.upper() or 'hek293' in l.lower()]
    print(f"Found {len(hek_lines)} HEK293-related entries:")
    for line in hek_lines:
        print(f"  {line}")
else:
    print("Sample table not yet downloaded.")

Found 2 HEK293-related entries:
  embryonic kidney cell line: HEK293/SLAM untreated	CNhs11046
  embryonic kidney cell line: HEK293/SLAM infection, 24hr	CNhs11047


In [6]:
print("=== FANTOM5 Peak Annotation ===")
download_file(
    url="https://fantom.gsc.riken.jp/5/datafiles/reprocessed/hg19_latest/extra/CAGE_peaks/hg19_fair+new_CAGE_peaks_phase1and2_ann.txt.gz",
    dest=RAW_DIR / "fantom5_peaks_annotation.txt.gz",
    description="FANTOM5 peak annotation file"
)

=== FANTOM5 Peak Annotation ===
  Already exists: fantom5_peaks_annotation.txt.gz (4,581,181 bytes)


In [7]:
print("=== FANTOM5 Expression Matrix ===")
print("NOTE: This is a large file (~4GB compressed). Download may take several minutes.")
download_file(
    url="https://fantom.gsc.riken.jp/5/datafiles/reprocessed/hg19_latest/extra/CAGE_peaks/hg19_fair+new_CAGE_peaks_phase1and2_tpm.osc.txt.gz",
    dest=RAW_DIR / "fantom5_cage_tpm_matrix.osc.txt.gz",
    description="FANTOM5 CAGE TPM expression matrix (~4GB compressed)"
)

=== FANTOM5 Expression Matrix ===
NOTE: This is a large file (~4GB compressed). Download may take several minutes.
  Already exists: fantom5_cage_tpm_matrix.osc.txt.gz (824,157,884 bytes)


## 4. DepMap — Cancer Dependency Data

Gene essentiality classifications from the Cancer Dependency Map.
- **Common Essential genes** — essential in most cell lines
- **Strongly Selective genes** — essential in a significant subset of cell lines
- **Gene dependency scores** — per cell line dependency probabilities (for counting dependent lines)

Starting with DepMap Public 24Q2 release. If distributions don't match Jason's reference,
we'll try 24Q4 and 25Q1.

In [8]:
print("=== DepMap Gene Lists (24Q2) ===")

# Common Essential controls
download_file(
    url="https://depmap.org/portal/download/all/?release=DepMap+Public+24Q2&file=AchillesCommonEssentialControls.csv",
    dest=RAW_DIR / "depmap_common_essential.csv",
    description="DepMap Common Essential gene list"
)

# Strongly Selective controls
download_file(
    url="https://depmap.org/portal/download/all/?release=DepMap+Public+24Q2&file=AchillesStronglySelectiveControls.csv",
    dest=RAW_DIR / "depmap_strongly_selective.csv",
    description="DepMap Strongly Selective gene list"
)

# Gene dependency scores (for counting dependent cell lines per gene)
print("\nNOTE: CRISPRGeneDependency.csv is large (~200MB). Download may take a minute.")
download_file(
    url="https://depmap.org/portal/download/all/?release=DepMap+Public+24Q2&file=CRISPRGeneDependency.csv",
    dest=RAW_DIR / "depmap_gene_dependency.csv",
    description="DepMap CRISPR Gene Dependency scores"
)

=== DepMap Gene Lists (24Q2) ===
  Already exists: depmap_common_essential.csv (15,952 bytes)
  Already exists: depmap_strongly_selective.csv (15,952 bytes)

NOTE: CRISPRGeneDependency.csv is large (~200MB). Download may take a minute.
  Already exists: depmap_gene_dependency.csv (15,952 bytes)


## 5. IMPC + MGI — Embryonic Lethality & Orthologs

- **IMPC** — International Mouse Phenotyping Consortium embryonic lethality data
- **MGI** — Mouse Genome Informatics human-mouse ortholog mapping

The IMPC provides viability classifications: Early lethal, Late lethal, Intermediate lethal.
MGI provides the ortholog table to map mouse genes to human genes.

In [9]:
print("=== MGI Human-Mouse Orthologs ===")
download_file(
    url="https://www.informatics.jax.org/downloads/reports/HOM_MouseHumanSequence.rpt",
    dest=RAW_DIR / "mgi_human_mouse_orthologs.rpt",
    description="MGI human-mouse ortholog mapping"
)

=== MGI Human-Mouse Orthologs ===
  Already exists: mgi_human_mouse_orthologs.rpt (15,106,108 bytes)


In [10]:
print("=== IMPC Embryonic Lethality Data ===")
# The IMPC genotype-phenotype assertions file contains all phenotype data
# including viability/lethality classifications.
# Try the latest release first.
try:
    download_file(
        url="https://ftp.ebi.ac.uk/pub/databases/impc/all-data-releases/latest/results/genotype-phenotype-assertions-ALL.csv.gz",
        dest=RAW_DIR / "impc_genotype_phenotype_all.csv.gz",
        description="IMPC genotype-phenotype assertions (all)"
    )
except Exception as e:
    print(f"  Failed to download from 'latest': {e}")
    print("  Trying alternative: IMPC viability data from web API...")
    # Fallback: try a specific release
    download_file(
        url="https://ftp.ebi.ac.uk/pub/databases/impc/all-data-releases/release-21.0/results/genotype-phenotype-assertions-ALL.csv.gz",
        dest=RAW_DIR / "impc_genotype_phenotype_all.csv.gz",
        description="IMPC genotype-phenotype assertions (release 21.0)"
    )

=== IMPC Embryonic Lethality Data ===
  Already exists: impc_genotype_phenotype_all.csv.gz (5,001,223 bytes)


## Download Manifest

Record all downloaded files with URLs, sizes, and download date.

In [11]:
# Generate download manifest
manifest_lines = [
    f"# Download Manifest\n",
    f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n",
    f"\n## Files\n\n",
]

sources = {
    "mmc3.xlsx": "https://ars.els-cdn.com/content/image/1-s2.0-S1097276517306512-mmc3.xlsx",
    "gencode.v19.annotation.gtf.gz": "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_19/gencode.v19.annotation.gtf.gz",
    "fantom5_sample_table.txt": "https://fantom.gsc.riken.jp/5/datafiles/latest/extra/Enhancers/Human.sample_name2library_id.txt",
    "fantom5_peaks_annotation.txt.gz": "https://fantom.gsc.riken.jp/5/datafiles/reprocessed/hg19_latest/extra/CAGE_peaks/hg19_fair+new_CAGE_peaks_phase1and2_ann.txt.gz",
    "fantom5_cage_tpm_matrix.osc.txt.gz": "https://fantom.gsc.riken.jp/5/datafiles/reprocessed/hg19_latest/extra/CAGE_peaks/hg19_fair+new_CAGE_peaks_phase1and2_tpm.osc.txt.gz",
    "depmap_common_essential.csv": "https://depmap.org/portal/download/all/?release=DepMap+Public+24Q2&file=AchillesCommonEssentialControls.csv",
    "depmap_strongly_selective.csv": "https://depmap.org/portal/download/all/?release=DepMap+Public+24Q2&file=AchillesStronglySelectiveControls.csv",
    "depmap_gene_dependency.csv": "https://depmap.org/portal/download/all/?release=DepMap+Public+24Q2&file=CRISPRGeneDependency.csv",
    "mgi_human_mouse_orthologs.rpt": "https://www.informatics.jax.org/downloads/reports/HOM_MouseHumanSequence.rpt",
    "impc_genotype_phenotype_all.csv.gz": "https://ftp.ebi.ac.uk/pub/databases/impc/all-data-releases/latest/results/genotype-phenotype-assertions-ALL.csv.gz",
}

for fname, url in sources.items():
    fpath = RAW_DIR / fname
    if fpath.exists():
        size = fpath.stat().st_size
        manifest_lines.append(f"| `{fname}` | {size:,} bytes | {url} |\n")
    else:
        manifest_lines.append(f"| `{fname}` | **NOT DOWNLOADED** | {url} |\n")

manifest_path = RAW_DIR / "MANIFEST.md"
with open(manifest_path, 'w') as f:
    f.writelines(manifest_lines)

print(f"Manifest saved to {manifest_path}")

Manifest saved to ../data/raw/MANIFEST.md


## Summary

Verify all files are present and report sizes.

In [12]:
print("=== Download Summary ===")
print(f"{'File':<50} {'Size':>15} {'Status'}")
print("-" * 80)

all_present = True
for fname in sources:
    fpath = RAW_DIR / fname
    if fpath.exists():
        size_mb = fpath.stat().st_size / (1024 * 1024)
        print(f"{fname:<50} {size_mb:>12.1f} MB  OK")
    else:
        print(f"{fname:<50} {'':>12}     MISSING")
        all_present = False

print("-" * 80)
if all_present:
    print("All files downloaded successfully.")
else:
    print("WARNING: Some files are missing. Check errors above.")

=== Download Summary ===
File                                                          Size Status
--------------------------------------------------------------------------------
mmc3.xlsx                                                   5.5 MB  OK
gencode.v19.annotation.gtf.gz                              36.2 MB  OK
fantom5_sample_table.txt                                    0.1 MB  OK
fantom5_peaks_annotation.txt.gz                             4.4 MB  OK
fantom5_cage_tpm_matrix.osc.txt.gz                        786.0 MB  OK
depmap_common_essential.csv                                 0.0 MB  OK
depmap_strongly_selective.csv                               0.0 MB  OK
depmap_gene_dependency.csv                                  0.0 MB  OK
mgi_human_mouse_orthologs.rpt                              14.4 MB  OK
impc_genotype_phenotype_all.csv.gz                          4.8 MB  OK
--------------------------------------------------------------------------------
All files downloaded successf